# ARGUS Kaggle: 58-Day Resumable Pipeline

This notebook runs days 1-58 with checkpoint-safe resume behavior and `/kaggle/working` persistence.

In [ ]:
# --- Parameters ---
AUTH_PATH = None  # Auto-detected from /kaggle/input unless manually set to an existing file
OUTPUT_ROOT = "/kaggle/working/argus_outputs"
START_DAY = 1
END_DAY = 58
DAY_BATCH_SIZE = 3  # Safe default; larger batches reduce repeated file rereads and run faster if the runtime stays alive
BUCKET_COUNT = 256  # Higher bucket_count lowers peak RAM in build_sessions.py
BUCKET_WORKERS = 2  # Kaggle usually benefits from a small amount of parallel bucket processing
TOKENIZE_PARQUET_BATCH_SIZE = 20000  # Larger batches reduce parquet overhead during tokenization
TOKENIZED_CHUNK_SIZE = 20000  # Larger chunks write far fewer .pt files; old 5000-row chunks still resume
TOKENIZE_MAX_LEN = 16  # Kaggle disk-safe sequence length; 512 is far too large for 100M+ sessions
TOKEN_ID_DTYPE = "int16"  # Use int32 if tokenization says the vocab is too large for int16
ATTENTION_MASK_DTYPE = "bool"
RESUME = True

# Repo setup options
USE_GIT_CLONE = True
REFRESH_GIT_CLONE = True  # Re-clone by default so Kaggle does not reuse a stale build_sessions.py
REPO_URL = "https://github.com/NIghtIngale340/ARGUS"
REPO_DIR = "/kaggle/working/ARGUS"
REPO_DATASET_DIR = "/kaggle/input/<argus-repo-dataset>/ARGUS"


In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

repo_dir = Path(REPO_DIR)
if repo_dir.exists() and USE_GIT_CLONE and REFRESH_GIT_CLONE:
    print(f"Refreshing git clone at: {repo_dir}")
    shutil.rmtree(repo_dir)

if repo_dir.exists():
    print(f"Using existing repo at: {repo_dir}")
elif USE_GIT_CLONE:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    src_dir = Path(REPO_DATASET_DIR)
    if not src_dir.exists():
        raise FileNotFoundError(f"REPO_DATASET_DIR not found: {src_dir}")
    shutil.copytree(src_dir, repo_dir)

if not (repo_dir / "scripts" / "build_sessions.py").exists():
    nested = repo_dir / "argus-log-intelligence-platform"
    if (nested / "scripts" / "build_sessions.py").exists():
        repo_dir = nested
        REPO_DIR = str(repo_dir)

BUILD_SESSIONS_SCRIPT = repo_dir / "scripts" / "build_sessions.py"

def ensure_build_sessions_supports_bucket_count() -> Path:
    if not BUILD_SESSIONS_SCRIPT.exists():
        raise FileNotFoundError(f"build_sessions.py not found at: {BUILD_SESSIONS_SCRIPT}")

    probe = subprocess.run(
        [sys.executable, str(BUILD_SESSIONS_SCRIPT), "--help"],
        cwd=str(repo_dir),
        capture_output=True,
        text=True,
        check=False,
    )
    help_text = (probe.stdout or "") + (probe.stderr or "")
    if probe.returncode != 0:
        raise RuntimeError(
            "Failed to inspect build_sessions.py CLI. "
            f"script={BUILD_SESSIONS_SCRIPT} returncode={probe.returncode}\n{help_text.strip()}"
        )
    if "--bucket-count" not in help_text or "--bucket-workers" not in help_text:
        raise RuntimeError(
            "Resolved build_sessions.py does not support the required bucket CLI flags. "
            f"The Kaggle repo copy is stale: {BUILD_SESSIONS_SCRIPT}. "
            "Refresh /kaggle/working/ARGUS or rerun the repo setup cell with REFRESH_GIT_CLONE=True."
        )

    print(f"Validated build_sessions.py CLI: {BUILD_SESSIONS_SCRIPT}")
    return BUILD_SESSIONS_SCRIPT

print(f"Repo ready: {repo_dir}")


In [ ]:
import subprocess
import sys

# Install only the pipeline dependencies needed for sessionization/tokenization in this notebook.
subprocess.run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "pandas",
    "pyarrow",
    "tqdm",
    "drain3",
    "jsonpickle",
    "orjson",
    "torch",
], check=True)
print("Pipeline dependencies installed.")


In [ ]:
from pathlib import Path

if AUTH_PATH and Path(AUTH_PATH).is_file():
    chosen = Path(AUTH_PATH)
    print(f"Using user-provided AUTH_PATH: {chosen}")
    print(f"AUTH_PATH size: {chosen.stat().st_size:,} bytes")
else:
    input_root = Path("/kaggle/input")
    matches = [p for p in input_root.rglob("auth.txt") if p.is_file()] if input_root.exists() else []
    if not matches:
        available = sorted([p.name for p in input_root.iterdir() if p.is_dir()]) if input_root.exists() else []
        raise FileNotFoundError(
            "Could not find a file named auth.txt under /kaggle/input. "
            f"Attach the LANL dataset first. Available input dirs: {available}"
        )

    matches = sorted(matches, key=lambda p: p.stat().st_size, reverse=True)
    chosen = matches[0]

    # Defensive guard: ensure selected path is a regular file before passing to build_sessions.py
    if not chosen.is_file():
        raise RuntimeError(f"Resolved AUTH_PATH is not a file: {chosen}")

    AUTH_PATH = str(chosen)
    print(f"Resolved AUTH_PATH: {AUTH_PATH}")
    print(f"AUTH_PATH size: {chosen.stat().st_size:,} bytes")
    if len(matches) > 1:
        print(f"Found {len(matches)} auth.txt files. Using largest: {AUTH_PATH}")


In [ ]:
# Optional smoke test before full-data run.
# Set RUN_SMOKE_TEST = True to validate environment quickly.
RUN_SMOKE_TEST = False

import subprocess
import sys
from pathlib import Path

if RUN_SMOKE_TEST:
    build_sessions_script = ensure_build_sessions_supports_bucket_count()
    sample_input = Path(REPO_DIR) / "data" / "raw" / "auth_sample_200k.txt"
    smoke_root = Path(OUTPUT_ROOT) / "smoke_test"
    (smoke_root / "data" / "sessions").mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable,
        str(build_sessions_script),
        "--input",
        str(sample_input),
        "--start-day",
        "1",
        "--end-day",
        "1",
        "--output-dir",
        str(smoke_root / "data" / "sessions"),
        "--parser-state",
        str(smoke_root / "data" / "drain3_state.bin"),
        "--bucket-count",
        str(BUCKET_COUNT),
        "--bucket-workers",
        str(BUCKET_WORKERS),
    ]
    print(f"Smoke test script: {build_sessions_script}")
    subprocess.run(cmd, check=True)
    print("Smoke test complete.")
else:
    print("Skipping smoke test.")

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import pyarrow.parquet as pq

output_root = Path(OUTPUT_ROOT)
data_root = output_root / "data"
sessions_dir = data_root / "sessions"
tokenized_dir = data_root / "tokenized"

sessions_dir.mkdir(parents=True, exist_ok=True)
tokenized_dir.mkdir(parents=True, exist_ok=True)
os.chdir(output_root)

def shard_ready(day: int) -> bool:
    shard = sessions_dir / f"day_{day:02d}.parquet"
    if not shard.exists() or shard.stat().st_size <= 0:
        return False
    try:
        pq.ParquetFile(shard)
    except Exception as exc:
        print(f"[WARN] day {day:02d} shard is unreadable and will be rebuilt: {exc}")
        shard.unlink(missing_ok=True)
        return False
    return True

def existing_session_shards():
    return sorted(
        p for p in sessions_dir.glob("day_*.parquet")
        if p.is_file() and p.stat().st_size > 0
    )

build_sessions_script = ensure_build_sessions_supports_bucket_count()
print(f"Using build_sessions.py: {build_sessions_script}")

ready_days = [day for day in range(START_DAY, END_DAY + 1) if shard_ready(day)]
not_ready_days = [day for day in range(START_DAY, END_DAY + 1) if day not in set(ready_days)]
print(f"Ready session shards: {len(ready_days)}/{END_DAY - START_DAY + 1}")
if not_ready_days:
    print(f"Days to build/rebuild: {not_ready_days[:20]}{' ...' if len(not_ready_days) > 20 else ''}")

batch_size = max(1, int(DAY_BATCH_SIZE))
day = START_DAY

while day <= END_DAY:
    shard = sessions_dir / f"day_{day:02d}.parquet"
    if RESUME and shard_ready(day):
        print(f"[SKIP] day {day:02d} already exists ({shard.stat().st_size} bytes)")
        day += 1
        continue

    batch_start = day
    batch_end = min(day + batch_size - 1, END_DAY)

    if RESUME:
        cursor = batch_start
        while cursor <= batch_end and not shard_ready(cursor):
            cursor += 1
        batch_end = cursor - 1

    if batch_end < batch_start:
        day += 1
        continue

    print(f"[RUN ] building days {batch_start:02d}-{batch_end:02d} (bucket_count={BUCKET_COUNT}, bucket_workers={BUCKET_WORKERS})")
    cmd = [
        sys.executable,
        str(build_sessions_script),
        "--input",
        AUTH_PATH,
        "--start-day",
        str(batch_start),
        "--end-day",
        str(batch_end),
        "--output-dir",
        "data/sessions",
        "--parser-state",
        "data/drain3_state.bin",
        "--bucket-count",
        str(BUCKET_COUNT),
        "--bucket-workers",
        str(BUCKET_WORKERS),
    ]
    subprocess.run(cmd, check=True)
    day = batch_end + 1

print("Session shard pass complete.")


## Checkpoint Step (Kaggle)

After each successful day batch, click **Save Version** in Kaggle.
Use a message like `processed day_01 to day_05` so resume state is easy to track.

In [ ]:
from pathlib import Path

sessions_dir = Path(OUTPUT_ROOT) / "data" / "sessions"
completed = sorted(p for p in sessions_dir.glob("day_*.parquet") if p.is_file() and p.stat().st_size > 0)
print(f"Completed non-empty shard count: {len(completed)}")
if completed:
    print(f"First shard: {completed[0].name}")
    print(f"Latest shard: {completed[-1].name}")
print("Session checkpoint check complete; continue to tokenization.")


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

os.chdir(OUTPUT_ROOT)

session_shards = sorted(
    p for p in (Path(OUTPUT_ROOT) / "data" / "sessions").glob("day_*.parquet")
    if p.is_file() and p.stat().st_size > 0
)
if not session_shards:
    raise FileNotFoundError(
        "No existing non-empty session parquet shards found at "
        f"{Path(OUTPUT_ROOT) / 'data' / 'sessions' / 'day_*.parquet'}"
    )

print(f"Tokenizing from {len(session_shards)} existing session parquet shard(s).")

split_jobs = [
    ("train", "--vocab-out", "data/vocab.json", "data/tokenized/sessions_train.pt", ["--reuse-vocab-if-exists"]),
    ("val", "--vocab-in", "data/vocab.json", "data/tokenized/sessions_val.pt", []),
    ("test", "--vocab-in", "data/vocab.json", "data/tokenized/sessions_test.pt", []),
]

for split, vocab_arg, vocab_path, token_out, extra_args in split_jobs:
    print(f"[RUN ] tokenization split={split}")
    cmd = [
        sys.executable,
        str(Path(REPO_DIR) / "scripts" / "build_vocab_and_tokenize.py"),
        "--sessions-glob",
        "data/sessions/day_*.parquet",
        "--split",
        split,
        vocab_arg,
        vocab_path,
        "--tokenized-out",
        token_out,
        "--parquet-batch-size",
        str(TOKENIZE_PARQUET_BATCH_SIZE),
        "--tokenized-chunk-size",
        str(TOKENIZED_CHUNK_SIZE),
        "--max-len",
        str(TOKENIZE_MAX_LEN),
        "--token-id-dtype",
        TOKEN_ID_DTYPE,
        "--attention-mask-dtype",
        ATTENTION_MASK_DTYPE,
        "--resume-tokenized",
        "--progress-log",
        f"data/tokenized/progress_{split}.jsonl",
    ]
    cmd.extend(extra_args)
    result = subprocess.run(cmd, capture_output=True, text=True, check=False)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print("STDERR:")
        print(result.stderr)
    result.check_returncode()

print("Tokenization pass complete.")


In [ ]:
from pathlib import Path

sessions_dir = Path(OUTPUT_ROOT) / "data" / "sessions"
tokenized_dir = Path(OUTPUT_ROOT) / "data" / "tokenized"

session_files = sorted(sessions_dir.glob("day_*.parquet"))
print(f"Session shards present: {len(session_files)}")
if session_files:
    print(f"First shard: {session_files[0].name}")
    print(f"Last shard:  {session_files[-1].name}")

for path in [
    Path(OUTPUT_ROOT) / "data" / "vocab.json",
    tokenized_dir / "sessions_train.pt",
    tokenized_dir / "sessions_val.pt",
    tokenized_dir / "sessions_test.pt",
]:
    print(f"{path}: exists={path.exists()} size={path.stat().st_size if path.exists() else 0}")

for chunk_dir in sorted(tokenized_dir.glob("sessions_*_chunks")):
    chunk_count = len(list(chunk_dir.glob("chunk_*.pt")))
    print(f"{chunk_dir}: chunks={chunk_count}")